# ORC1 Fiber Assay Analysis

**Purpose:** Compare replication speed and inter-origin distance between WT and MGS variants.

**Author:** Elena Lopatukhina
**Date:** 2026-07-14

## Workflow
1. Libraries import
2. Parameters import
3. Data loading
4. Data propcessing
5. Basic statistics calculation
6. Processing outliers
7. Statistical analysis
8. Tables export

# 1. Libraries import

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, ttest_ind
import warnings

# 2. Parameters

In [1]:
INPUT_DIR = "/mnt/c/users/helen/Desktop/test"
OUTPUT_DIR = "/mnt/c/users/helen/Desktop/test"

pixel_size = 0.16125 # µm
conversion_factor = 2.59 # kb/µm
time = 20 # minutes

# 3. Import data

## 3.1 Data loading

In [2]:
from utils import load_data, data_subset

In [3]:
data = load_data(dir = INPUT_DIR, pixel_size=pixel_size)

print(f"Total number of measurements for analysis is: {data.shape[0]}")

Total number of measurements for analysis is: 39


In [4]:
data.head()

,Sample_name,File,Measurement_type,Length,ROI,Path
0,None,HCl_1h15min_15o_Fiber_length,Fiber_length,9.623723,0464-0881,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...
1,None,HCl_1h15min_15o_Fiber_length,Fiber_length,8.880521,0495-0819,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...
2,None,HCl_1h15min_15o_Fiber_length,Fiber_length,8.624940,0464-0300,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...
3,None,HCl_1h15min_15o_Fiber_length,Fiber_length,3.212906,0448-0336,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...
4,None,HCl_1h15min_15o_Fiber_length,Fiber_length,9.381686,0396-0436,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...


### Manually fill Sample_name column

In [5]:
# Create Sample name column empty (custom parsing of filenames)
data['Sample_name'] = data['File'].apply(lambda x: x.split("_")[1].split("-")[0])

data["Sample_name"] = data["Sample_name"].apply(
    lambda x: "WT" if "1h15min" in x else x
)

data["Sample_name"] = data["Sample_name"].apply(
    lambda x: "WT" if "30min" in x else x
)

mask = (
    (data["Sample_name"] == "HaloEmpty") &
    (data["File"].str.contains("siORC1_HaloEmpty", na=False))
)
data.loc[mask, "Sample_name"] = "siORC1"

# Update only rows where Sample_name == "HaloEmpty" and File contains "siSCR_HaloEmpty"
mask = (
    (data["Sample_name"] == "HaloEmpty") &
    (data["File"].str.contains("siSCR_HaloEmpty", na=False))
)

data.loc[mask, "Sample_name"] = "siSCR"

# Info about samples
sample_names = set(data["Sample_name"])
print(f"The amount of samples is: {len(sample_names)}.")
print(f"There are: {sample_names}")

The amount of samples is: 2.
There are: {'siSCR', 'WT'}


In [6]:
data.head()

,Sample_name,File,Measurement_type,Length,ROI,Path
0,WT,HCl_1h15min_15o_Fiber_length,Fiber_length,9.623723,0464-0881,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...
1,WT,HCl_1h15min_15o_Fiber_length,Fiber_length,8.880521,0495-0819,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...
2,WT,HCl_1h15min_15o_Fiber_length,Fiber_length,8.624940,0464-0300,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...
3,WT,HCl_1h15min_15o_Fiber_length,Fiber_length,3.212906,0448-0336,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...
4,WT,HCl_1h15min_15o_Fiber_length,Fiber_length,9.381686,0396-0436,/mnt/c/users/helen/Desktop/test/HCl_1h15min_15...


### Split data into replication speed and IOD dataframe

In [8]:
speed = data_subset(df = data, measurement_type = 'Fiber_length')
print(f"The total amount of fibers measurements is: {speed.shape[0]}")

iod = data_subset(df = data, measurement_type = 'Interorigin_distance')
print(f"The total amount of IOD measurements is: {iod.shape[0]}")

The total amount of fibers measurements is: 30
The total amount of IOD measurements is: 9


## 3.2 Checking the number of measurements for each sample

### 3.2.1 Replication speed
Divide this number by 2 because these are green and red tracks separately.

In [ ]:
speed.groupby('Sample_name')['File'].count()

In [ ]:
# OPTIONAL
# Check analized files
#sample_name = "MGS5"
#speed[speed["Sample_name"] == sample_name]['File'].value_counts()

### 3.2.2. IOD

In [ ]:
iod.groupby('Sample_name')['File'].count()

In [ ]:
# OPTIONAL
# Check analized files
#sample_name = "siORC1"
#iod[iod["Sample_name"] == sample_name]['File'].value_counts()

# 4. Data processing

## 4.1. Replication speed data processing

In [ ]:
# Checking speed file
counts = speed.groupby("File").size()
odd_files = counts[counts % 2 != 0].index.tolist()

if len(odd_files) == 0:
    print("All files contain an even number of fibers.")
else:
    print("The following files contain an odd number of fibers will be removed:")
    print(*odd_files, sep="\n")
    
    # Removing odd files from speed dataframe
    speed = speed[~speed["File"].isin(odd_files)].copy()

In [ ]:
# Add extra inedex to group pairs of files
speed["Index"] = speed.groupby("File").cumcount() // 2

# Calculate sum of fiber length in pairs
speed_processed = speed.groupby(["File", "Index"], as_index=False).agg(
        Total_Length=("Length", "sum"),
        ROI=("ROI", list),
        Path=("Path", "first"),
        Sample_name=("Sample_name", "first")
        )

# Convert speed to kb/min
speed_processed['Speed_kb_min'] = speed_processed['Total_Length'].apply(lambda x: x * conversion_factor / time)

# Delete extra columns
replication_speed = speed_processed[['Sample_name', 'File', 'Speed_kb_min', 'ROI', 'Path']]

In [ ]:
replication_speed.head()

## 4.2 IOD data processing

In [ ]:
iod['IOD_kb'] = iod['Length'].apply(lambda x: x * conversion_factor)
iod_kb = iod[["Sample_name", "File", 'IOD_kb', 'ROI', 'Path']]

iod_kb.head()

# 5. Basic statistics calculation

## 5.1 Replication speed statistics

In [ ]:
stats_speed = (
    replication_speed.groupby("Sample_name")["Speed_kb_min"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        SD="std",
    )
)

stats_speed

## 5.2 IOD statistics

In [ ]:
stats_iod = (
    iod_kb.groupby("Sample_name")["IOD_kb"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        SD="std",
    )
)

stats_iod

# 6. Processing outliers

## 6.1 Replication speed outliers

In [ ]:
Q1 = replication_speed['Speed_kb_min'].quantile(0.25)
Q3 = replication_speed['Speed_kb_min'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_speed = replication_speed[(replication_speed['Speed_kb_min'] < lower) | (replication_speed['Speed_kb_min'] > upper)]

outliers_speed

## 6.2 IOD outliers

In [ ]:
Q1 = iod_kb['IOD_kb'].quantile(0.25)
Q3 = iod_kb['IOD_kb'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_iod = iod_kb[(iod_kb['IOD_kb'] < lower) | (iod_kb['IOD_kb'] > upper)]

outliers_iod

# 7. Statistical analysis
Mann-Whetney test (U-test)

### 7.1 Replication speed analysis

In [ ]:
# -------------------------------------------------------
# Data and settings
# -------------------------------------------------------
data_plot = replication_speed
var = "Speed_kb_min"
control_sample = "WT"

# Subset of control data to compare with
control_df = data_plot.loc[
    data_plot["Sample_name"] == control_sample,
    var
].dropna()

experimental_samples = set(data_plot["Sample_name"]) - {control_sample}

results = []

# Iteration through the experimental samples
for sample in experimental_samples:
    
    # Subset of experimental sample
    sample_df = data_plot.loc[
    data_plot["Sample_name"] == sample,
    var
    ].dropna()
    
    # Statistics calculation
    stat, p = mannwhitneyu(
    control_df,
    sample_df
    )
    
    results.append({
    "Group_1": control_sample,
    "Group_2": sample,
    "Group_1_n": len(control_df),
    "Group_2_n": len(sample_df),
    "U": stat,
    "p-value": p,
    })

# Convert data into dataframe
speed_u = pd.DataFrame(results)

# Show results
speed_u

### 7.2 IOD analysis

In [ ]:
# -------------------------------------------------------
# Data and settings
# -------------------------------------------------------
data_plot = iod_kb
var = "IOD_kb"
control_sample = "WT"

# Subset of control data to compare with
control_df = data_plot.loc[
    data_plot["Sample_name"] == control_sample,
    var
].dropna()

experimental_samples = set(data_plot["Sample_name"]) - {control_sample}

results = []

# Iteration through the experimental samples
for sample in experimental_samples:
    
    # Subset of experimental sample
    sample_df = data_plot.loc[
    data_plot["Sample_name"] == sample,
    var
    ].dropna()
    
    # Statistics calculation
    stat, p = mannwhitneyu(
    control_df,
    sample_df
    )
    
    results.append({
    "Group_1": control_sample,
    "Group_2": sample,
    "Group_1_n": len(control_df),
    "Group_2_n": len(sample_df),
    "U": stat,
    "p-value": p,
    })

# Convert data into dataframe
iod_u = pd.DataFrame(results)

# Show results
iod_u

# 8. Tables export

In [ ]:
# -------------------------------------------------------
# Export of tables with processed results
# -------------------------------------------------------
replication_speed.to_excel(f"{OUTPUT_DIR}/replication_speed.xlsx", index=False)
iod_kb.to_excel(f"{OUTPUT_DIR}/iod_kb.xlsx", index=False)

# -------------------------------------------------------
# Export of tables with descriptive statistics
# -------------------------------------------------------
stats_speed.to_excel(f"{OUTPUT_DIR}/replication_speed_description.xlsx", index=False)
stats_iod.to_excel(f"{OUTPUT_DIR}/iod_description.xlsx", index=False)

# -------------------------------------------------------
# Export of tables with U-test analysis
# -------------------------------------------------------
speed_u.to_excel(f"{OUTPUT_DIR}/stats_u_replication_speed.xlsx", index=False)
iod_u.to_excel(f"{OUTPUT_DIR}/stats_u_iod.xlsx", index=False)

# 9. Graphs with statistics

In [ ]:
from plotting import boxplot_with_statistics

In [ ]:
boxplot_with_statistics(data_plot = replication_speed,
                            sample_order = ['siSCR', 'WT'],
                            var = 'Speed_kb_min',
                            stats_plot = speed_u,
                            y_axis = "Replication speed, kb/min",
                            save_dir = "./examples",
                            save_name = "speed",
                            ext = "png")

In [ ]:
boxplot_with_statistics(data_plot = iod_kb,
                            sample_order = ['siSCR', 'WT'],
                            var = 'IOD_kb',
                            stats_plot = iod_u,
                            y_axis = "IOD, kb",
                            save_dir = "./examples",
                            save_name = "iod",
                            ext = "png")

### *Optional code*

In [ ]:
# -------------------------------------------------------
# t-test data analysis
# -------------------------------------------------------

# Data
# data_plot = iod_kb
# var = "IOD_kb"

# wt = data_plot.loc[
    # data_plot["Sample_name"] == "WT",
    # var
# ].dropna()

# results = []

# for sample in ["siSCR", "siORC1", "MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]:

    # mutant = data_plot.loc[
        # data_plot["Sample_name"] == sample,
        # var
    # ].dropna()

    # stat, p = ttest_ind(
        # wt,
        # mutant,
        # equal_var=False,   # Welch's t-test (recommended)
    # )

    # results.append({
        # "Comparison": f"WT vs {sample}",
        # "WT_n": len(wt),
        # "Sample_n": len(mutant),
        # "U": stat,
        # "p-value": p,
    # })

# stats_t_df = pd.DataFrame(results)

# print(stats_t_df)

In [ ]:
# -------------------------------------------------------
# Boxplot without statistical brackets
# -------------------------------------------------------

# plt.figure(figsize=(7, 5))

# Variables
# data_plot = iod_kb
# var = "IOD_kb"

# Order of groups (optional)
# sample_order = ["siSCR", "siORC1", "WT", "MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]

# groups = []
# labels = []

# for sample in sample_order:
        # values = data_plot.loc[
        # data_plot["Sample_name"] == sample,
        # var
    # ]
        
        # groups.append(values)
        # labels.append(sample)

    

# bp = plt.boxplot(
    # groups,
    # patch_artist=True,
    # showfliers=False,
    # widths=0.6,
# )

# for box in bp["boxes"]:
    # box.set(facecolor="#4C72B0", alpha=0.6)

# Jittered dots
# for i, values in enumerate(groups, start=1):
    # x = np.random.normal(i, 0.05, len(values))
    # plt.scatter(x, values, s=20, color="black", alpha=0.7, zorder=3)

# plt.ylabel("iod_kb")
# plt.xticks(range(1, len(labels) + 1), labels)
# plt.grid(axis="y", linestyle="--", alpha=0.4)

# plt.tight_layout()
# plt.savefig(f"{OUTPUT_DIR}/iod_boxplot.png", dpi=600, bbox_inches="tight")
# print(f"Plot is saved in the directory: {OUTPUT_DIR}")

# plt.show()